# Three loose ends from step 9 (added after looking)

Step 9 and the check on new markets left three results marked "not looked into". This notebook looks, using their saved predictions. Nothing is rerun.

1. Why does set D gain more in North Asia (+9 to +13 points) than in ASEAN (+5 to +9)?
2. Why does set A beat the fair bar open to close in Indonesia?
3. Why is AdaBoost with set A below the fair bar open to close in Thailand?

In [1]:
import numpy as np
import pandas as pd

from data import MARKETS, NEW_MARKETS, load_all, load_us
from features import make_us_features, up_open_to_close
from more_inputs import MODELS, SAVED

def saved(target, s, model, market):
    folder = SAVED / "new_markets" if market in NEW_MARKETS else SAVED
    return pd.read_pickle(folder / f"{s}_{target}_{model}_{market}.pkl".replace(" ", "_"))

def accuracy(p, calls=None):
    calls = p["p_up"] > 0.5 if calls is None else calls
    return 100 * (calls == p["up"]).mean()

markets = load_all(include_holdout=True, markets=MARKETS + NEW_MARKETS)
us = load_us(include_holdout=True)

## 1. Set D's gain against how closely each market follows the US

Set D's gain over A (close to close, the mean over the seven models), and the correlation of the last US session's move with day t's move.

In [2]:
rows = {}
for m, df in markets.items():
    us_move = make_us_features(df, us)["us_ret"]
    rows[m] = {
        "D gain over A, points": np.mean([accuracy(saved("close to close", "D", model, m)) - accuracy(saved("close to close", "A", model, m))
                                          for model in MODELS]),
        "US move with day t": us_move.corr(df["ret"]),
    }
follows_us = pd.DataFrame(rows).T
print("correlation of the two columns over the 10 markets:", round(follows_us.iloc[:, 0].corr(follows_us.iloc[:, 1]), 2))
follows_us.sort_values("US move with day t").round(2)

correlation of the two columns over the 10 markets: 0.93


,"D gain over A, points",US move with day t
Thailand,4.92,0.24
Indonesia,5.87,0.24
India,5.89,0.27
Hong Kong,7.50,0.28
Philippines,7.48,0.30
Korea,10.45,0.31
Malaysia,7.94,0.33
Singapore,9.20,0.37
Taiwan,11.48,0.41
Japan,12.95,0.43


## 2 and 3. Indonesia and Thailand, open to close, set A

Each model's gain over the fair bar by test year. The fair bar is the more common direction in **all** training years. For comparison, a rule with no model: **last year's** more common direction.

In [3]:
def last_year_rule(m):
    """Each test day's guess: the direction that was more common, open to close, in the year before."""
    up = up_open_to_close(markets[m]).dropna().astype(bool)
    share_up = up.groupby(up.index.year).mean()
    p = saved("open to close", "A", "LDA", m)  # the days that were scored
    return pd.Series(share_up.reindex(p["year"] - 1).to_numpy() >= 0.5, index=p.index)

for m in ["Indonesia", "Thailand"]:
    rows = {}
    for model in MODELS:
        p = saved("open to close", "A", model, m)
        rows[model] = {**{y: accuracy(f) - accuracy(f, f["fair_call"]) for y, f in p.groupby("year")},
                       "all years": accuracy(p) - accuracy(p, p["fair_call"]), "called up %": 100 * (p["p_up"] > 0.5).mean()}
    p, rule = saved("open to close", "A", "LDA", m), last_year_rule(m)
    rows["last year's direction (no model)"] = {**{y: accuracy(f, rule[f.index]) - accuracy(f, f["fair_call"]) for y, f in p.groupby("year")},
                                                "all years": accuracy(p, rule) - accuracy(p, p["fair_call"]), "called up %": 100 * rule.mean()}
    share = 100 * p.groupby("year")["up"].mean()
    print(f"{m}: fair bar says {'up' if p['fair_call'].all() else 'down' if not p['fair_call'].any() else 'both'} every year. "
          f"Share of days up, by year: {share.round(0).astype(int).to_dict()}")
    display(pd.DataFrame(rows).T.round(1))

Indonesia: fair bar says up every year. Share of days up, by year: {2017: 50, 2018: 52, 2019: 47, 2020: 46, 2021: 46, 2022: 54, 2023: 53, 2024: 48, 2025: 44, 2026: 45}


,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,all years,called up %
LDA,0.0,0.4,-0.8,-3.1,10.5,-0.5,1.9,4.3,8.8,2.6,2.8,66.5
Logistic regression,0.8,2.9,2.0,-2.3,-0.4,1.6,-1.9,4.3,6.1,3.6,2.1,72.2
KNN,-0.4,0.8,-0.4,3.9,6.9,6.6,-1.0,4.3,8.8,7.8,3.8,76.4
SVM,0.0,0.0,0.0,0.0,0.0,7.1,-1.0,2.1,6.6,4.7,2.0,86.8
Random forest,1.7,1.2,6.1,1.6,6.5,0.5,-1.9,4.3,6.1,4.7,3.5,66.7
AdaBoost,2.5,2.9,-0.8,0.8,8.9,3.8,2.9,4.7,3.1,9.3,3.9,71.7
Neural network,3.4,1.7,4.1,0.8,7.7,-5.5,1.0,2.6,7.9,3.6,3.1,48.3
last year's direction (no model),0.0,0.0,0.0,8.5,8.5,-7.1,0.0,0.0,11.4,10.9,3.2,51.9


Thailand: fair bar says down every year. Share of days up, by year: {2017: 45, 2018: 47, 2019: 45, 2020: 45, 2021: 41, 2022: 50, 2023: 41, 2024: 42, 2025: 42, 2026: 49}


,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,all years,called up %
LDA,-10.3,-3.3,-1.6,-1.6,-3.7,-0.8,-4.5,-2.5,0.0,2.6,-2.7,30.5
Logistic regression,-7.4,-2.4,-0.4,-4.9,0.0,0.4,0.4,-3.3,0.0,2.1,-1.6,18.4
KNN,-2.9,-1.2,-1.6,0.4,0.4,1.2,-2.5,-7.8,1.7,-0.5,-1.3,21.1
SVM,-1.6,-1.2,-2.0,1.6,0.8,0.0,-2.1,-3.7,-0.4,0.0,-0.9,20.3
Random forest,0.4,1.2,-2.0,-3.3,-3.3,-2.9,-6.6,-13.2,-3.7,0.0,-3.4,33.1
AdaBoost,-6.6,0.0,-4.9,-3.3,-5.4,0.0,-13.6,-1.2,-0.4,0.0,-3.6,23.5
Neural network,-8.6,3.7,-2.0,-2.9,-5.4,0.8,-8.6,1.2,0.0,1.5,-2.1,25.0
last year's direction (no model),-9.5,0.0,0.0,0.0,0.0,0.0,-17.7,0.0,0.0,0.0,-2.8,20.4


## Conclusions

1. **Set D gains most where a market follows the US most.** Over the 10 markets, D's gain and the US move's correlation with day t go together (0.93). Japan (0.43) and Taiwan (0.41) follow the US most and gain most (+13.0 and +11.5 points); Thailand and Indonesia (0.24) least (+4.9 and +5.9). North Asia isn't special beyond that.
2. **Indonesia's open-to-close "edge" is mostly a drift, not a forecast.** The fair bar says up every year, but days rose after the open on only 44–48% of days in 2019–2021 and 2024–2026. A model that says up less often gains then. Following last year's direction, with no model at all, gains +3.2 points, about as much as the models (+2.0 to +3.9). The years only partly match (both gain in 2021, 2025 and 2026, but not in 2020 and 2022), so this explains much of it, not all.
3. **Thailand's AdaBoost result is the same drift, going the other way, not a bug.** Days rose after the open on only 41–50% of days, so the fair bar says down every year and any "up" call is costly. All seven models are below it (−0.9 to −3.6 points), and so is following last year's direction (−2.8). AdaBoost is the worst, mostly from 2017 and 2023 (−6.6 and −13.6), the same two years in which following last year's direction loses most (−9.5 and −17.7).

**In short:** none of the three is a hidden edge or a bug. One is how much a market follows the US, and two are how often a market's days rise after the open, which drifts from year to year.